# 02 — Xây dựng và mô tả nhân tố Fama–French Việt Nam

Notebook này trình bày các nhân tố MKT−RF, SMB, HML, RMW và CMA đã được dựng bằng portfolio sort trên VN100. Nó kiểm tra đủ sáu danh mục thành phần cho từng phép sort, sau đó tái tạo bảng thống kê, ma trận tương quan và biểu đồ lợi suất tích lũy.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir()
OUT = ROOT / 'outputs' / 'vn_period_factors'

## Công thức

Các cổ phiếu được chia theo quy mô và đặc trưng vào 2×3 danh mục. HML là trung bình hai danh mục High trừ trung bình hai danh mục Low; RMW và CMA được tính tương tự. SMB của FF3 lấy trung bình chênh lệch Small−Big trong sort B/M; SMB của FF5 là trung bình SMB từ ba sort B/M, profitability và investment.

In [ ]:
factors = pd.read_csv(OUT / 'vn100_factors_monthly.csv', parse_dates=['Date'])
summary = pd.read_csv(OUT / 'vn100_factor_summary.csv', index_col=0)
correlation = pd.read_csv(OUT / 'vn100_factor_correlations.csv', index_col=0)
groups = pd.read_csv(OUT / 'vn100_group_counts.csv')

display(summary)
display(correlation.round(3))

In [ ]:
sort_check = (
    groups.groupby(['period', 'sort'])
    .agg(portfolios=('portfolio', 'nunique'), min_stocks=('ticker_count', 'min'), total_stocks=('ticker_count', 'sum'))
    .reset_index()
)
display(sort_check)
assert (sort_check['portfolios'] == 6).all(), 'Có sort không đủ sáu danh mục'
assert (sort_check['min_stocks'] > 0).all(), 'Có danh mục thành phần rỗng'

## Lợi suất tích lũy của các nhân tố

Mỗi đường được chuẩn hóa về 1 tại tháng đầu tiên mà nhân tố có dữ liệu; vì vậy RMW và CMA có thể bắt đầu muộn hơn SMB và HML.

In [ ]:
plot_cols = ['SMB', 'HML', 'RMW', 'CMA']
fig, ax = plt.subplots(figsize=(10, 5.5))
for col in plot_cols:
    series = factors.set_index('Date')[col].dropna()
    ((1 + series).cumprod()).plot(ax=ax, label=col, linewidth=2)
ax.axhline(1, color='black', linewidth=0.8, alpha=0.5)
ax.set(title='Lợi suất tích lũy của các nhân tố VN100', xlabel='Tháng', ylabel='Giá trị tích lũy (khởi đầu = 1)')
ax.grid(alpha=0.25)
ax.legend(ncol=4)
plt.tight_layout()
plt.show()

Số liệu dùng cho bảng báo cáo nằm trực tiếp trong `vn100_factor_summary.csv`; notebook không làm tròn trước khi tính toán, chỉ làm tròn khi hiển thị.